In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()

np.random.seed(42)
random.seed(42)

In [2]:
NUM_USERS = 5000

start_date = pd.to_datetime("2023-01-01")
end_date = pd.to_datetime("2023-12-31")

date_range = pd.date_range(start_date, end_date)

In [3]:
campaign_ids = [f"CMP_{i}" for i in range(1, 11)]

channels = ["Facebook", "Google", "Instagram", "Organic", "Referral"]

In [4]:
users = []

for i in range(NUM_USERS):
    user_id = f"USR_{i+1}"

    signup_date = np.random.choice(date_range)

    campaign_id = np.random.choice(campaign_ids)

    acquisition_channel = np.random.choice(channels, p=[0.25, 0.25, 0.2, 0.2, 0.1])

    age = np.random.randint(18, 60)

    gender = np.random.choice(["Male", "Female"])

    country = np.random.choice(["India", "USA", "UK", "Canada"])

    # KYC logic (80% complete)
    kyc_completed = np.random.choice([1, 0], p=[0.8, 0.2])

    if kyc_completed:
        kyc_date = signup_date + pd.Timedelta(days=np.random.randint(1, 5))
    else:
        kyc_date = None

    users.append([
        user_id,
        signup_date,
        country,
        age,
        gender,
        acquisition_channel,
        campaign_id,
        kyc_completed,
        kyc_date
    ])

users_df = pd.DataFrame(users, columns=[
    "user_id",
    "signup_date",
    "country",
    "age",
    "gender",
    "acquisition_channel",
    "campaign_id",
    "kyc_completed",
    "kyc_date"
])

In [19]:
import os

os.makedirs("data/raw", exist_ok=True)

users_df.to_csv("data/raw/users.csv", index=False)

In [14]:
users_df.head()
users_df.info()
users_df['kyc_completed'].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              5000 non-null   str           
 1   signup_date          5000 non-null   datetime64[us]
 2   country              5000 non-null   str           
 3   age                  5000 non-null   int64         
 4   gender               5000 non-null   str           
 5   acquisition_channel  5000 non-null   str           
 6   campaign_id          5000 non-null   str           
 7   kyc_completed        5000 non-null   int64         
 8   kyc_date             3983 non-null   datetime64[us]
dtypes: datetime64[us](2), int64(2), str(5)
memory usage: 351.7 KB


kyc_completed
1    3983
0    1017
Name: count, dtype: int64

In [15]:
users_df.head()

,user_id,signup_date,country,age,gender,acquisition_channel,campaign_id,kyc_completed,kyc_date
0,USR_1,2023-04-13,India,25,Male,Referral,CMP_4,1,2023-04-16
1,USR_2,2023-08-03,UK,57,Female,Google,CMP_8,1,2023-08-05
2,USR_3,2023-12-10,Canada,38,Male,Facebook,CMP_6,1,2023-12-11
3,USR_4,2023-08-24,Canada,59,Female,Google,CMP_9,0,NaT
4,USR_5,2023-09-28,UK,20,Male,Google,CMP_3,1,2023-09-29


In [16]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              5000 non-null   str           
 1   signup_date          5000 non-null   datetime64[us]
 2   country              5000 non-null   str           
 3   age                  5000 non-null   int64         
 4   gender               5000 non-null   str           
 5   acquisition_channel  5000 non-null   str           
 6   campaign_id          5000 non-null   str           
 7   kyc_completed        5000 non-null   int64         
 8   kyc_date             3983 non-null   datetime64[us]
dtypes: datetime64[us](2), int64(2), str(5)
memory usage: 351.7 KB


In [17]:
users_df['kyc_completed'].value_counts(normalize=True)

kyc_completed
1    0.7966
0    0.2034
Name: proportion, dtype: float64

In [18]:
(users_df['kyc_date'] >= users_df['signup_date']).sum()

np.int64(3983)

In [21]:
users_df = pd.read_csv("../data/raw/users.csv", parse_dates=["signup_date", "kyc_date"])

Transaction Data Generation

In [22]:
transactions = []

transaction_id_counter = 1

transaction_types = ["deposit", "withdrawal", "payment"]
status_options = ["success", "failed"]

In [23]:
for _, row in users_df.iterrows():
    
    user_id = row["user_id"]
    signup_date = row["signup_date"]
    kyc_completed = row["kyc_completed"]
    
    # Users without KYC → very low chance of transactions
    if kyc_completed == 0:
        num_transactions = np.random.choice([0, 1], p=[0.9, 0.1])
    else:
        # Active vs inactive users
        num_transactions = np.random.choice(
            [0, 5, 10, 20, 50], 
            p=[0.1, 0.3, 0.3, 0.2, 0.1]
        )
    
    for _ in range(num_transactions):
        
        transaction_date = signup_date + pd.Timedelta(days=np.random.randint(1, 180))
        
        amount = round(np.random.uniform(100, 10000), 2)
        
        transaction_type = np.random.choice(transaction_types)
        
        status = np.random.choice(status_options, p=[0.9, 0.1])
        
        transactions.append([
            f"TXN_{transaction_id_counter}",
            user_id,
            transaction_date,
            amount,
            transaction_type,
            status
        ])
        
        transaction_id_counter += 1

transactions_df = pd.DataFrame(transactions, columns=[
    "transaction_id",
    "user_id",
    "transaction_date",
    "amount",
    "transaction_type",
    "status"
])

In [24]:
transactions_df.to_csv("../data/raw/transactions.csv", index=False)

Validate

In [25]:
transactions_df.head()

,transaction_id,user_id,transaction_date,amount,transaction_type,status
0,TXN_1,USR_1,2023-07-04,5432.95,withdrawal,success
1,TXN_2,USR_1,2023-09-19,1535.62,withdrawal,success
2,TXN_3,USR_1,2023-06-24,8325.37,withdrawal,success
3,TXN_4,USR_1,2023-08-16,1092.02,payment,success
4,TXN_5,USR_1,2023-06-08,8055.74,payment,success


In [26]:
transactions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54163 entries, 0 to 54162
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    54163 non-null  str           
 1   user_id           54163 non-null  str           
 2   transaction_date  54163 non-null  datetime64[us]
 3   amount            54163 non-null  float64       
 4   transaction_type  54163 non-null  str           
 5   status            54163 non-null  str           
dtypes: datetime64[us](1), float64(1), str(4)
memory usage: 2.5 MB


In [27]:
len(transactions_df)

54163

Check relationship integrity

In [28]:
transactions_df["user_id"].isin(users_df["user_id"]).all()

np.True_

Check date logic

In [29]:
merged = transactions_df.merge(users_df, on="user_id")

(merged["transaction_date"] >= merged["signup_date"]).all()

np.True_

Check failure rate

In [30]:
transactions_df["status"].value_counts(normalize=True)

status
success    0.898972
failed     0.101028
Name: proportion, dtype: float64

Campaigns Data Generation

In [31]:
campaigns = []

channels = ["Facebook", "Google", "Instagram", "Email", "Referral"]

for i in range(1, 11):
    
    campaign_id = f"CMP_{i}"
    
    campaign_name = f"Campaign_{i}"
    
    channel = np.random.choice(channels)
    
    start_date = pd.to_datetime("2023-01-01") + pd.Timedelta(days=np.random.randint(0, 200))
    
    end_date = start_date + pd.Timedelta(days=np.random.randint(10, 60))
    
    cost = round(np.random.uniform(10000, 100000), 2)
    
    campaigns.append([
        campaign_id,
        campaign_name,
        channel,
        start_date,
        end_date,
        cost
    ])

campaigns_df = pd.DataFrame(campaigns, columns=[
    "campaign_id",
    "campaign_name",
    "channel",
    "start_date",
    "end_date",
    "cost"
])

In [32]:
campaigns_df.to_csv("../data/raw/campaigns.csv", index=False)

Validation

In [33]:
campaigns_df

,campaign_id,campaign_name,channel,start_date,end_date,cost
0,CMP_1,Campaign_1,Google,2023-01-03,2023-01-13,76341.67
1,CMP_2,Campaign_2,Facebook,2023-04-14,2023-05-08,68436.70
2,CMP_3,Campaign_3,Google,2023-04-12,2023-06-03,30484.32
3,CMP_4,Campaign_4,Google,2023-06-13,2023-07-04,30298.24
4,CMP_5,Campaign_5,Google,2023-01-15,2023-03-09,53812.12
5,CMP_6,Campaign_6,Google,2023-01-30,2023-03-11,78916.57
6,CMP_7,Campaign_7,Referral,2023-02-16,2023-03-07,90570.94
7,CMP_8,Campaign_8,Email,2023-02-13,2023-03-09,54606.78
8,CMP_9,Campaign_9,Google,2023-04-05,2023-05-02,97215.85
9,CMP_10,Campaign_10,Google,2023-02-25,2023-04-22,28976.99


In [34]:
campaigns_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   campaign_id    10 non-null     str           
 1   campaign_name  10 non-null     str           
 2   channel        10 non-null     str           
 3   start_date     10 non-null     datetime64[us]
 4   end_date       10 non-null     datetime64[us]
 5   cost           10 non-null     float64       
dtypes: datetime64[us](2), float64(1), str(3)
memory usage: 612.0 bytes


Check uniqueness

In [35]:
campaigns_df["campaign_id"].nunique()

10

Check date logic

In [36]:
(campaigns_df["end_date"] > campaigns_df["start_date"]).all()

np.True_

Load all dataset

In [37]:
users_df = pd.read_csv("../data/raw/users.csv", parse_dates=["signup_date", "kyc_date"])
transactions_df = pd.read_csv("../data/raw/transactions.csv", parse_dates=["transaction_date"])
campaigns_df = pd.read_csv("../data/raw/campaigns.csv", parse_dates=["start_date", "end_date"])

Basic Cleaning

In [38]:
# Remove duplicates (safety)
users_df = users_df.drop_duplicates()
transactions_df = transactions_df.drop_duplicates()
campaigns_df = campaigns_df.drop_duplicates()

# Ensure correct data types
users_df["kyc_completed"] = users_df["kyc_completed"].astype(int)

transactions_df["amount"] = transactions_df["amount"].astype(float)

Feature Engineering

First Transaction per User

In [39]:
first_txn = transactions_df[transactions_df["status"] == "success"] \
    .groupby("user_id")["transaction_date"] \
    .min() \
    .reset_index()

first_txn.rename(columns={"transaction_date": "first_transaction_date"}, inplace=True)

users_df = users_df.merge(first_txn, on="user_id", how="left")

User-Level Metrics

In [40]:
txn_summary = transactions_df[transactions_df["status"] == "success"] \
    .groupby("user_id") \
    .agg(
        total_transactions=("transaction_id", "count"),
        total_amount=("amount", "sum"),
        avg_transaction_value=("amount", "mean")
    ) \
    .reset_index()

users_df = users_df.merge(txn_summary, on="user_id", how="left")

Fill Nulls

In [41]:
users_df["total_transactions"] = users_df["total_transactions"].fillna(0)
users_df["total_amount"] = users_df["total_amount"].fillna(0)
users_df["avg_transaction_value"] = users_df["avg_transaction_value"].fillna(0)

KPI Flags

In [42]:
# Converted user (did at least 1 successful transaction)
users_df["is_converted"] = users_df["total_transactions"].apply(lambda x: 1 if x > 0 else 0)

# Active user (more than 5 transactions)
users_df["is_active_user"] = users_df["total_transactions"].apply(lambda x: 1 if x >= 5 else 0)

Funnel Stage Flags

In [43]:
users_df["kyc_done_flag"] = users_df["kyc_completed"]

users_df["transaction_done_flag"] = users_df["is_converted"]

Cohort Column

In [44]:
users_df["cohort_month"] = users_df["signup_date"].dt.to_period("M").astype(str)

Save Processed Data

In [45]:
import os
os.makedirs("../data/processed", exist_ok=True)

users_df.to_csv("../data/processed/users_cleaned.csv", index=False)
transactions_df.to_csv("../data/processed/transactions_cleaned.csv", index=False)
campaigns_df.to_csv("../data/processed/campaigns_cleaned.csv", index=False)

FINAL VALIDATION

In [46]:
users_df.head()

,user_id,signup_date,country,age,gender,acquisition_channel,campaign_id,kyc_completed,kyc_date,first_transaction_date,total_transactions,total_amount,avg_transaction_value,is_converted,is_active_user,kyc_done_flag,transaction_done_flag,cohort_month
0,USR_1,2023-04-13,India,25,Male,Referral,CMP_4,1,2023-04-16,2023-05-23,10.0,44526.59,4452.659000,1,1,1,1,2023-04
1,USR_2,2023-08-03,UK,57,Female,Google,CMP_8,1,2023-08-05,2023-09-13,5.0,29532.43,5906.486000,1,1,1,1,2023-08
2,USR_3,2023-12-10,Canada,38,Male,Facebook,CMP_6,1,2023-12-11,2023-12-23,48.0,226313.42,4714.862917,1,1,1,1,2023-12
3,USR_4,2023-08-24,Canada,59,Female,Google,CMP_9,0,NaT,NaT,0.0,0.00,0.000000,0,0,0,0,2023-08
4,USR_5,2023-09-28,UK,20,Male,Google,CMP_3,1,2023-09-29,NaT,0.0,0.00,0.000000,0,0,1,0,2023-09


In [47]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   user_id                 5000 non-null   str           
 1   signup_date             5000 non-null   datetime64[us]
 2   country                 5000 non-null   str           
 3   age                     5000 non-null   int64         
 4   gender                  5000 non-null   str           
 5   acquisition_channel     5000 non-null   str           
 6   campaign_id             5000 non-null   str           
 7   kyc_completed           5000 non-null   int64         
 8   kyc_date                3983 non-null   datetime64[us]
 9   first_transaction_date  3681 non-null   datetime64[us]
 10  total_transactions      5000 non-null   float64       
 11  total_amount            5000 non-null   float64       
 12  avg_transaction_value   5000 non-null   float64       
 13 

In [48]:
users_df[["is_converted", "is_active_user"]].value_counts()

is_converted  is_active_user
1             1                 3089
0             0                 1319
1             0                  592
Name: count, dtype: int64